In [275]:
from pathlib import Path
from typing import Final

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import tsam

# Entry Point
This notebook starts from cleaned hourly input datasets, validates them, reshapes data for the method that we are going to use, runs TSAM aggregation, merges the reduced dataset together and inspects the outputs.

## On the custom method
The method includes the following steps:
- split the year by real calendar month
- classify each day as working or non-working
- cluster each month/day-type group separately
- select real observed days as representatives
- assign each representative day a weight equal to the number of original days in its cluster
- concatenate selected representative days into a constructed reduced chronology

### The expected structure:
- 24 clustering groups: 12 months x 2 day types
- 84 representative days: 12 x (5 + 2)
- 2016 hourly snapshots: 84 x 24
- 12 monthly reduced blocks: each 7 x 24 = 168 hours

## Table of Contents

- [Entry Point](#entry-point)
- [Data Path Config](#data-path-config)
- [Data Loading](#data-loading)
- [Snapshot Feature Handling](#snapshot-feature-handling)
- [Data Validation](#data-validation)
  - [Duplicates, NaNs, Numeric Columns & Full-Year Coverage](#duplicates-nans-numeric-columns--full-year-coverage)
  - [CF Range Validation](#cf-range-validation)
  - [Running Validation](#running-validation)
- [TSAM Clustering](#tsam-clustering)
- [Looking At The Results](#looking-at-the-results)
  - [Shapes](#shapes)
  - [Outputs](#outputs)
  - [Extremes Comparison](#extremes-comparison)
- [Plotting Results](#plotting-results)
  - [Cluster Representatives](#cluster-representatives)
  - [Cluster Members](#cluster-members)
  - [Cluster Weights](#cluster-weights)
  - [Cluster Accuracy](#cluster-accuracy)
  - [Full-Resolution Comparison](#full-resolution-comparison)
  - [Residuals](#residuals)
  - [Calendar Heatmap Of Original Days By Assigned Cluster](#calendar-heatmap-of-original-days-by-assigned-cluster)
  - [Month-By-Cluster Heatmap](#month-by-cluster-heatmap)
- [Saving Results](#saving-results)


# Data Path Config


In [276]:
# Notebook working directory. In this project, running from src/ makes ../data resolve to data/.
cur_location: Path = Path().resolve()
data_location: Path = cur_location / "../data"

# Data Loading


Set the correct CSV separator with `sep`. In these datasets, some files use `,` and others use `;`.


In [277]:
demand_EU = pd.read_csv(data_location / "Demand_ENTSO_E.csv", sep=";")
demand_EU.head()

,snapshot,AL,AT,BA,BE,BG,CH,CZ,DE,DK,...,NL,NO,PL,PT,RO,RS,SE,SI,SK,UK
0,01.01.2025 00:00,459,3871,609,6691,2374,4603,4029,28833,2606,...,9796,11525,11852,3591,3645,2578,9701,728,1823,18285
1,01.01.2025 01:00,428,3717,571,6327,2292,4711,3989,27781,2511,...,9535,11586,11284,3480,3532,2474,9551,699,1780,18154
2,01.01.2025 02:00,396,3522,539,6128,2237,4861,3885,26917,2426,...,9266,11540,10983,3289,3467,2316,9377,666,1733,16545
3,01.01.2025 03:00,376,3488,523,5982,2206,4810,3855,27016,2400,...,9013,11517,10827,3126,3438,2175,9417,648,1729,14918
4,01.01.2025 04:00,367,3598,526,5967,2194,4725,3865,26870,2405,...,9040,11602,10754,3007,3428,2059,9560,664,1724,14200


In [278]:
onwind_cf = pd.read_csv(data_location / "onwind_capacity_factors.csv", sep=",")
onwind_cf.head()

,snapshot,AL_onwind_2025,AT_onwind_2025,BA_onwind_2025,BE_onwind_2025,BG_onwind_2025,CH_onwind_2025,CZ_onwind_2025,DE_onwind_2025,DK_onwind_2025,...,NO_onwind_2025,PL_onwind_2025,PT_onwind_2025,RO_onwind_2025,RS_onwind_2025,SE_onwind_2025,SI_onwind_2025,SK_onwind_2025,UA_onwind_2025,XK_onwind_2025
0,01.01.2025 00:00,0.0,0.026443,0.0,0.968038,0.027777,0.030767,0.419450,0.716840,1.000000,...,0.131214,0.873780,0.116221,0.051165,0.0,0.297268,0.013709,0.114870,0.346401,0.0
1,01.01.2025 01:00,0.0,0.029090,0.0,0.975868,0.027310,0.030301,0.426459,0.728613,0.999992,...,0.109802,0.897004,0.109594,0.054343,0.0,0.286711,0.015315,0.114240,0.345689,0.0
2,01.01.2025 02:00,0.0,0.033465,0.0,0.983856,0.028493,0.029116,0.443365,0.743509,0.999972,...,0.092031,0.911984,0.105639,0.062091,0.0,0.278833,0.019344,0.110237,0.332158,0.0
3,01.01.2025 03:00,0.0,0.034560,0.0,0.988063,0.030633,0.029512,0.467903,0.760476,0.999996,...,0.078412,0.923292,0.107288,0.067657,0.0,0.271188,0.023585,0.110385,0.326958,0.0
4,01.01.2025 04:00,0.0,0.036894,0.0,0.987477,0.034158,0.031685,0.495288,0.771249,1.000000,...,0.071422,0.927705,0.103116,0.072680,0.0,0.259012,0.025262,0.112671,0.322901,0.0


In [279]:
hydro_inflow = pd.read_csv(
    data_location / "hydro_inflow_scaled_2025.csv", sep=";"
)
hydro_inflow.head()

,snapshot,AT_hydro_2025,BA_hydro_2025,BE_hydro_2025,BG_hydro_2025,CH_hydro_2025,CZ_hydro_2025,DE_hydro_2025,ES_hydro_2025,FI_hydro_2025,...,RO_hydro_2025.1,RS_hydro_2025,RS_hydro_2025.1,RS_hydro_2025.2,SI_hydro_2025,UA_hydro_2025,UK_hydro_2025,XK_hydro_2025,SE_hydro_2025,AL_hydro_2025
0,01.01.2025 00:00,230,83,2,49,411,39,54,922,420,...,56,2,2,8,135,556,43,3,4570,155
1,01.01.2025 01:00,229,83,2,49,411,40,54,922,420,...,56,2,2,8,135,555,43,3,4569,152
2,01.01.2025 02:00,229,83,2,49,412,41,54,922,421,...,56,2,2,8,135,555,43,3,4562,151
3,01.01.2025 03:00,228,83,2,49,412,41,54,922,422,...,56,2,2,8,135,555,43,3,4555,151
4,01.01.2025 04:00,228,83,2,49,412,41,54,922,423,...,56,2,2,8,135,555,43,3,4545,150


In [280]:
ror_cf = pd.read_csv(data_location / "ror_capacity_factors.csv", sep=",")
ror_cf.head()

,snapshot,AL_ror_2025,AT_ror_2025,BA_ror_2025,BE_ror_2025,BG_ror_2025,CH_ror_2025,CZ_ror_2025,DE_ror_2025,EE_ror_2025,...,NO_ror_2025,PL_ror_2025,PT_ror_2025,RO_ror_2025,RS_ror_2025,SE_ror_2025,SI_ror_2025,SK_ror_2025,UK_ror_2025,XK_ror_2025
0,01.01.2025 00:00,0.06198,0.304260,0.113561,0.687213,0.132907,0.266242,0.128353,0.596077,0.0,...,0.592028,0.503166,0.140826,0.371941,0.428571,0.35421,0.090094,0.232892,0.19516,0.07694
1,01.01.2025 01:00,0.06074,0.300330,0.149166,0.687989,0.132323,0.264119,0.406737,0.591670,0.0,...,0.585834,0.503452,0.062204,0.369386,0.411529,0.35409,0.087915,0.231347,0.19512,0.07694
2,01.01.2025 02:00,0.06031,0.300783,0.155915,0.626787,0.129821,0.262051,0.430766,0.584802,0.0,...,0.578365,0.502625,0.059455,0.363477,0.338847,0.35360,0.087664,0.233002,0.19510,0.07682
3,01.01.2025 03:00,0.06016,0.306833,0.111967,0.609455,0.130377,0.260064,0.438508,0.577704,0.0,...,0.576665,0.502492,0.044515,0.359085,0.249123,0.35307,0.087212,0.230574,0.19506,0.07675
4,01.01.2025 04:00,0.05997,0.305347,0.111861,0.587503,0.135743,0.258158,0.438407,0.575100,0.0,...,0.580272,0.502750,0.096728,0.359085,0.246115,0.35229,0.087112,0.230022,0.19502,0.07684


In [281]:
solar_cf = pd.read_csv(data_location / "solar_capacity_factors.csv", sep=";")
solar_cf.head()

,snapshot,AL_solar_2025,AT_solar_2025,BA_solar_2025,BE_solar_2025,BG_solar_2025,CH_solar_2025,CZ_solar_2025,DE_solar_2025,DK_solar_2025,...,NO_solar_2025,PL_solar_2025,PT_solar_2025,RO_solar_2025,RS_solar_2025,SE_solar_2025,SI_solar_2025,SK_solar_2025,UA_solar_2025,XK_solar_2025
0,01.01.2025 00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,01.01.2025 01:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,01.01.2025 02:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,01.01.2025 03:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,01.01.2025 04:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


For this example, we use Germany-only data.


In [282]:
# Slice Germany-only columns for the walkthrough.
demand_DE = demand_EU[["snapshot", "DE"]]
solar_cf_DE = solar_cf[["snapshot", "DE_solar_2025"]]
onwind_cf_DE = onwind_cf[["snapshot", "DE_onwind_2025"]]
ror_cf_DE = ror_cf[["snapshot", "DE_ror_2025"]]
hydro_inflow_DE = hydro_inflow[["snapshot", "DE_hydro_2025"]]

# Name/data pairs used by the validation loop below.
data: list[tuple[str, pd.DataFrame]] = [
    ("demand_DE", demand_DE),
    ("solar_cf_DE", solar_cf_DE),
    ("onwind_cf_DE", onwind_cf_DE),
    ("ror_cf_DE", ror_cf_DE),
    ("hydro_inflow_DE", hydro_inflow_DE),
]

# Snapshot Feature Handling
TSAM expects snapshots to be the DataFrame index, and that index must be a datetime index.


In [283]:
def set_new_index(
    df: pd.DataFrame, new_index_col: str = "snapshot"
) -> pd.DataFrame:
    """Set ``new_index_col`` as the DataFrame index in-place and return ``df``.

    The notebook keeps separate variables like ``demand_DE`` and also stores the same
    DataFrame objects in ``data``. Mutating in-place keeps both references aligned.
    """
    if new_index_col not in df.columns:
        raise KeyError(
            f"{new_index_col!r} is not a column in the provided DataFrame"
        )

    df.set_index(new_index_col, drop=True, inplace=True)
    return df


# Set the snapshot column as the index for every source table.
data = [(df_name, set_new_index(df, "snapshot")) for df_name, df in data]

print("Index columns were swapped successfully!")

Index columns were swapped successfully!


Convert each index to datetime. The snapshots use day-first dates, so we pass `dayfirst=True`.


In [284]:
# Converting index to datetime type
for df_name, df in data:
    df.index = pd.to_datetime(df.index, dayfirst=True)

print("Successfully converted indexes into datetime dtype!")

Successfully converted indexes into datetime dtype!


# Data Validation
**Note:** You can skip this section and jump to TSAM clustering. If the input data has a problem, these checks will raise an error and explain what failed.

We validate the inputs before aggregation so TSAM does not fail silently or produce misleading representative periods.

Checks performed:
- no duplicate timestamps
- no NaNs
- numeric columns only
- complete hourly year coverage
- capacity-factor columns stay in the expected `0.0-1.0` range


## Duplicates, NaNs, Numeric Columns & Full-Year Coverage
This validator checks the main TSAM input requirements: datetime index, complete hourly coverage, no duplicates, no NaNs, and numeric columns only.


In [285]:
def validate_df(
    df: pd.DataFrame,
    name: str,
    year_to_check: int = 2025,
    period: str = "h",
) -> None:
    """Validate that a DataFrame is ready for hourly TSAM aggregation.

    TSAM expects a regular DatetimeIndex, numeric values, no gaps, no duplicates,
    and no NaNs. This function fails fast before aggregation can silently produce
    misleading representative periods.
    """
    expected_index = pd.date_range(
        start=f"{year_to_check}-01-01 00:00",
        end=f"{year_to_check}-12-31 23:00",
        freq=period,
    )

    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(f"{name}: index must be a DatetimeIndex")

    if len(df) != len(expected_index):
        raise ValueError(
            f"{name}: expected {len(expected_index)} rows, got {len(df)}"
        )

    if df.index.has_duplicates:
        raise ValueError(f"{name}: index has duplicate timestamps")

    if not df.index.equals(expected_index):
        missing = expected_index.difference(df.index)
        extra = df.index.difference(expected_index)
        raise ValueError(
            f"{name}: index does not exactly match hourly {year_to_check}. "
            f"Missing examples: {missing[:5].tolist()}. "
            f"Extra examples: {extra[:5].tolist()}."
        )

    if df.isna().any().any():
        cols = df.columns[df.isna().any()].tolist()
        raise ValueError(f"{name}: contains NaNs in columns {cols}")

    non_numeric_cols = df.select_dtypes(exclude="number").columns.tolist()
    if non_numeric_cols:
        raise TypeError(
            f"{name}: non-numeric columns found: {non_numeric_cols}"
        )


## CF Range Validation
Capacity-factor features should stay between `0.0` and `1.0`. This validator raises an error if any CF value falls outside that range.


In [286]:
def validate_cf_range(df: pd.DataFrame, name: str) -> None:
    """Validate that capacity-factor columns stay inside the physical 0..1 range."""
    is_in_range = df.ge(0.0) & df.le(1.0)

    if not is_in_range.all().all():
        invalid_cols = df.columns[~is_in_range.all()].tolist()
        raise ValueError(
            f"{name}: values outside the 0.0-1.0 CF range in {invalid_cols}"
        )


## Running Validation
Run all validation checks. If a check fails, the cell raises an error with the failing condition.


In [287]:
capacity_factor_datasets: Final[set[str]] = {
    "solar_cf_DE",
    "onwind_cf_DE",
    "ror_cf_DE",
}

for name, df in data:
    validate_df(df, name, year_to_check=2025, period="h")

    if name in capacity_factor_datasets:
        validate_cf_range(df, name)

print("Data validation has been successful!")

Data validation has been successful!


TSAM expects one dataset for the region being aggregated. Here, we join all Germany-specific features into one DataFrame.


In [288]:
data_DE: pd.DataFrame = demand_DE.join(
    [solar_cf_DE, onwind_cf_DE, ror_cf_DE, hydro_inflow_DE]
).rename(columns={"DE": "demand_DE"})

data_DE.head()

,demand_DE,DE_solar_2025,DE_onwind_2025,DE_ror_2025,DE_hydro_2025
snapshot,,,,,
2025-01-01 00:00:00,28833,0.0,0.716840,0.596077,54
2025-01-01 01:00:00,27781,0.0,0.728613,0.591670,54
2025-01-01 02:00:00,26917,0.0,0.743509,0.584802,54
2025-01-01 03:00:00,27016,0.0,0.760476,0.577704,54
2025-01-01 04:00:00,26870,0.0,0.771249,0.575100,54


# Building Daily Metadata
The Option 1 method clusters days separately by month and by working/non-working status. To do that, we add calendar metadata to the hourly Germany dataset.

There are two useful table grains in this section:

- `hourly_data_with_metadata`: hourly table with calendar metadata, used later for TSAM group slicing
- `daily_metadata`: one-row-per-day table, used only for calendar validation

## Metadata Creation
Create the calendar fields needed to form the 24 clustering groups:

- date
- month
- day of month
- weekday
- working/non-working label
- group ID, such as `2025_1_working` or `2025_1_non-working`

Assumptions:
- Saturday and Sunday are non-working.
- Holidays are excluded unless a holiday calendar is supplied.

In [289]:
# Baseline: weekends only, no holidays.
NON_WORKING_WEEKDAYS: Final[set[str]] = {"Saturday", "Sunday"}

# Hourly table + calendar metadata.
hourly_data_with_metadata = data_DE.copy()
snapshot_index = hourly_data_with_metadata.index

hourly_data_with_metadata["date"] = snapshot_index.date
hourly_data_with_metadata["month"] = snapshot_index.month
hourly_data_with_metadata["day_of_month"] = snapshot_index.day
hourly_data_with_metadata["weekday"] = snapshot_index.day_name()

# Default to working days, then override weekends.
hourly_data_with_metadata["day_type"] = "working"
hourly_data_with_metadata.loc[
    hourly_data_with_metadata["weekday"].isin(NON_WORKING_WEEKDAYS),
    "day_type",
] = "non-working"

# Key for the 24 TSAM groups, e.g. 2025_1_working.
hourly_data_with_metadata["group_id"] = (
    pd.Series(snapshot_index.year, index=snapshot_index).astype(str)
    + "_"
    + hourly_data_with_metadata["month"].astype(str)
    + "_"
    + hourly_data_with_metadata["day_type"]
)
hourly_data_with_metadata.head()

,demand_DE,DE_solar_2025,DE_onwind_2025,DE_ror_2025,DE_hydro_2025,date,month,day_of_month,weekday,day_type,group_id
snapshot,,,,,,,,,,,
2025-01-01 00:00:00,28833,0.0,0.716840,0.596077,54,2025-01-01,1,1,Wednesday,working,2025_1_working
2025-01-01 01:00:00,27781,0.0,0.728613,0.591670,54,2025-01-01,1,1,Wednesday,working,2025_1_working
2025-01-01 02:00:00,26917,0.0,0.743509,0.584802,54,2025-01-01,1,1,Wednesday,working,2025_1_working
2025-01-01 03:00:00,27016,0.0,0.760476,0.577704,54,2025-01-01,1,1,Wednesday,working,2025_1_working
2025-01-01 04:00:00,26870,0.0,0.771249,0.575100,54,2025-01-01,1,1,Wednesday,working,2025_1_working


## Metadata Validation
Validate the calendar split before running TSAM. A bad split can still produce clusters, but those clusters would represent the wrong kinds of days.

This section checks three things:

- weekday/weekend classification follows the baseline rule
- the day-level metadata covers the full 365-day year
- all 24 month/day-type groups are feasible for the requested cluster counts

In [290]:
# One row per original day for calendar validation.
daily_metadata = (
    hourly_data_with_metadata.assign(
        date=hourly_data_with_metadata.index.normalize()
    )
    .drop_duplicates(subset="date")
    .set_index("date")
)

weekday_order: Final[list[str]] = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]
working_weekdays: Final[list[str]] = [
    weekday for weekday in weekday_order if weekday not in NON_WORKING_WEEKDAYS
]
day_type_order: Final[list[str]] = ["working", "non-working"]

# Validate weekday/weekend classification.
weekday_day_type_counts = pd.crosstab(
    daily_metadata["weekday"], daily_metadata["day_type"]
).reindex(index=weekday_order, columns=day_type_order, fill_value=0)
weekday_day_type_counts

day_type,working,non-working
weekday,,
Monday,52,0
Tuesday,52,0
Wednesday,53,0
Thursday,52,0
Friday,52,0
Saturday,0,52
Sunday,0,52


The crosstab is the visual check: Monday-Friday should appear only under `working`, and Saturday/Sunday only under `non-working`.

The assertions below make the same rule executable and also confirm that no day was lost while collapsing hourly data to daily metadata.

In [291]:
# Weekday/weekend classification rule.
assert (weekday_day_type_counts.loc[working_weekdays, "non-working"] == 0).all()
assert (weekday_day_type_counts.loc[list(NON_WORKING_WEEKDAYS), "working"] == 0).all()

# Full non-leap year coverage.
expected_days_in_year: Final[int] = 365
total_days = len(daily_metadata)
assert total_days == expected_days_in_year, (
    f"The day-level metadata should contain {expected_days_in_year} days, got: {total_days}"
)

# Crosstab should account for the same day rows.
assert weekday_day_type_counts.to_numpy().sum() == total_days
print("The validation has been passed!")

The validation has been passed!


The final table checks the 24 groups used by the aggregation loop. Each month must have enough real working and non-working days to select the requested number of medoids.

In [292]:
# Requested representative days per month/day-type group.
requested_clusters_by_day_type: Final[dict[str, int]] = {
    "working": 5,
    "non-working": 2,
}
month_order: Final[range] = range(1, 13)

# Count real days in each of the 24 groups.
month_day_type_counts = (
    daily_metadata.groupby(["month", "day_type"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=month_order, columns=requested_clusters_by_day_type.keys(), fill_value=0)
)

# 12 months x 2 day types.
expected_group_count = len(month_order) * len(requested_clusters_by_day_type)
assert daily_metadata["group_id"].nunique() == expected_group_count

# Each group must have enough days for its requested clusters.
has_enough_days_for_clusters = month_day_type_counts.ge(
    pd.Series(requested_clusters_by_day_type),
    axis="columns",
)
assert has_enough_days_for_clusters.all().all(), (
    "Each month/day-type group must have at least as many days as requested clusters"
)

month_day_type_counts

day_type,working,non-working
month,,
1,23,8
2,20,8
3,21,10
4,22,8
5,22,9
6,21,9
7,23,8
8,21,10
9,22,8


# TSAM Clustering
Option 1 is not one annual TSAM run. It is 24 small TSAM runs: one for each month/day-type group.

For each group:

- pass only that group's hourly data into TSAM
- use 24-hour daily periods
- request 5 medoids for working-day groups and 2 medoids for non-working-day groups
- use medoid representation so selected days are real observed days
- skip extreme-day enrichment in this baseline

For each group we collect:

- reduced hourly rows for the selected medoid days
- original day => assigned cluster mapping
- cluster weights
- the raw TSAM result object for debugging

The combined result should have 84 representative days and 2016 hourly rows.

[API reference](https://tsam.readthedocs.io/en/latest/api/tsam/api/#tsam.api.aggregate)


## Data Collection
Start with the 24 group IDs created from the metadata section. Sorting them makes the notebook output deterministic.

In [293]:
group_ids = sorted(hourly_data_with_metadata["group_id"].unique())
group_ids

['2025_10_non-working',
 '2025_10_working',
 '2025_11_non-working',
 '2025_11_working',
 '2025_12_non-working',
 '2025_12_working',
 '2025_1_non-working',
 '2025_1_working',
 '2025_2_non-working',
 '2025_2_working',
 '2025_3_non-working',
 '2025_3_working',
 '2025_4_non-working',
 '2025_4_working',
 '2025_5_non-working',
 '2025_5_working',
 '2025_6_non-working',
 '2025_6_working',
 '2025_7_non-working',
 '2025_7_working',
 '2025_8_non-working',
 '2025_8_working',
 '2025_9_non-working',
 '2025_9_working']

These helpers prepare one group at a time.

They keep two versions of the same group:

- feature-only data passed into TSAM
- feature + metadata data used to map TSAM outputs back to original days

In [294]:
# Original feature columns.
FEATURE_COLUMNS = data_DE.columns
WORKING_CLUSTERS_PER_MONTH = 5
NON_WORKING_CLUSTERS_PER_MONTH = 2


def add_group_day_number(group_data_with_metadata: pd.DataFrame) -> pd.DataFrame:
    """Add a 1-based day number inside one month/day-type group."""
    group_data_with_metadata["group_day_number"] = (
        group_data_with_metadata["day_of_month"]
        .ne(group_data_with_metadata["day_of_month"].shift())
        .cumsum()
    )
    return group_data_with_metadata


def slice_group_data(
    group_id: str,
    data_with_metadata: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Slice one month/day-type group and return TSAM features plus metadata."""
    # Keep metadata for later mapping.
    group_data_with_metadata = data_with_metadata.loc[
        data_with_metadata["group_id"] == group_id
    ].copy()
    group_data_with_metadata = add_group_day_number(group_data_with_metadata)

    # TSAM input: numeric feature columns only.
    group_features = group_data_with_metadata.loc[:, FEATURE_COLUMNS]
    return group_features, group_data_with_metadata


def get_n_clusters_for_group(group_data_with_metadata: pd.DataFrame) -> int:
    """Return requested cluster count for one month/day-type group."""
    day_types = group_data_with_metadata["day_type"].unique()

    if len(day_types) != 1:
        raise ValueError(f"Expected one day_type per group, got: {day_types}")

    day_type = day_types[0]

    if day_type == "working":
        return WORKING_CLUSTERS_PER_MONTH
    elif day_type == "non-working":
        return NON_WORKING_CLUSTERS_PER_MONTH

    raise ValueError(f"Unknown day_type: {day_type}")


def collect_representative_day_data(
    aggregation_result: tsam.AggregationResult,
    group_data_with_metadata: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Build reduced hourly data and original-day assignments for one group."""
    representative_day_chunks = []
    # TSAM cluster centers are 0-based day positions inside this group.
    representative_day_indices = aggregation_result.clustering.cluster_centers
    cluster_weights_by_id = aggregation_result.cluster_weights

    # One row per original day: which cluster it belongs to.
    day_assignments = (
        aggregation_result.assignments.assign(
            date=aggregation_result.assignments.index.normalize()
        )
        .drop_duplicates("date")
        .set_index("date")
    )
    day_assignments = day_assignments.loc[:, ["cluster_idx"]].rename(
        columns={"cluster_idx": "cluster_id"}
    )
    day_assignments["cluster_weight"] = day_assignments["cluster_id"].map(
        lambda cluster_id: cluster_weights_by_id[cluster_id]
    )
    # Same cluster IDs repeat across groups, so keep the group key.
    day_assignments["group_id"] = group_data_with_metadata["group_id"]
    # Global representative ID: group key + local cluster ID.
    day_assignments["representative_id"] = (
        day_assignments["group_id"].astype(str)
        + "_c"
        + day_assignments["cluster_id"].astype(str)
    )

    cluster_ids = sorted(cluster_weights_by_id)
    for cluster_id, representative_day_index in zip(
        cluster_ids,
        representative_day_indices,
    ):
        # Slice the selected medoid day from the original group data.
        representative_day_hours = group_data_with_metadata.loc[
            group_data_with_metadata["group_day_number"]
            == representative_day_index + 1
        ].copy()

        representative_day_hours["cluster_id"] = cluster_id
        representative_day_hours["cluster_weight"] = cluster_weights_by_id[cluster_id]

        current_group_id = representative_day_hours["group_id"].iloc[0]
        representative_day_hours["representative_id"] = (
            f"{current_group_id}_c{cluster_id}"
        )

        representative_day_chunks.append(representative_day_hours)

    # Concatenate all representative days from this group.
    group_reduced_hourly_data = pd.concat(representative_day_chunks)
    return group_reduced_hourly_data, day_assignments

## Data Aggregation
Run TSAM once for each group and combine the per-group outputs.

Important settings:

- `period_duration=24`: each TSAM period is one day
- `representation="medoid"`: representatives are real observed days
- `preserve_column_means=False`: do not rescale medoid profiles away from the original observed values

The aggregation returns three objects:

- `reduced_hourly_df`: selected medoid days, one row per reduced hour
- `day_assignments_df`: one row per original day, showing its assigned cluster
- `tsam_results_by_group`: raw TSAM results keyed by group ID

In [295]:
BASELINE_CLUSTER_CONFIG = tsam.ClusterConfig(
    method="hierarchical",
    representation="medoid",
)
PERIOD_DURATION_HOURS: Final[int] = 24
TSAM_NUMERICAL_TOLERANCE: Final[float] = 1e-9


def run_aggregation_all_groups(
    group_ids: list[str],
    data_with_metadata: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, tsam.AggregationResult]]:
    """Run one TSAM aggregation per group and combine the outputs."""
    reduced_hourly_chunks: list[pd.DataFrame] = []
    day_assignment_chunks: list[pd.DataFrame] = []
    tsam_results_by_group: dict[str, tsam.AggregationResult] = {}

    for group_id in group_ids:
        # Slice this group into TSAM features and metadata-rich source data.
        group_features, group_data_with_metadata = slice_group_data(
            group_id,
            data_with_metadata,
        )

        # Working groups get 5 medoids; non-working groups get 2.
        n_clusters = get_n_clusters_for_group(group_data_with_metadata)

        aggregation_result = tsam.aggregate(
            data=group_features,
            n_clusters=n_clusters,
            period_duration=PERIOD_DURATION_HOURS,
            preserve_column_means=False,
            cluster=BASELINE_CLUSTER_CONFIG,
            numerical_tolerance=TSAM_NUMERICAL_TOLERANCE,
        )

        tsam_results_by_group[group_id] = aggregation_result

        group_reduced_hourly_data, group_day_assignments = (
            collect_representative_day_data(
                aggregation_result,
                group_data_with_metadata,
            )
        )
        reduced_hourly_chunks.append(group_reduced_hourly_data)
        day_assignment_chunks.append(group_day_assignments)

    reduced_hourly_data = pd.concat(reduced_hourly_chunks).sort_index()
    day_assignments_data = pd.concat(day_assignment_chunks).sort_index()

    return reduced_hourly_data, day_assignments_data, tsam_results_by_group

Apply the aggregation and inspect the reduced hourly table first. This table contains only selected medoid days, not a reconstructed 365-day year.

In [296]:
reduced_hourly_df, day_assignments_df, tsam_results_by_group = (
    run_aggregation_all_groups(group_ids, hourly_data_with_metadata)
)

Build a compact representative-day summary table. This has one row per selected medoid day, so the expected shape is 84 rows: `12 * (5 working + 2 non-working)`.

In [297]:
representative_day_columns: Final[list[str]] = [
    "selected_medoid_date",
    "representative_id",
    "group_id",
    "month",
    "day_type",
    "cluster_id",
    "cluster_weight",
]

representative_days = (
    reduced_hourly_df
    .drop_duplicates(subset="representative_id", keep="first")
    .rename(columns={"date": "selected_medoid_date"})
    .loc[:, representative_day_columns]
    .sort_values("selected_medoid_date")
    .reset_index(drop=True)
)

## Sanity Checks
These checks validate the structure of the reduced dataset before any plotting or downstream modelling.

They verify four things:

- expected object sizes: 2016 reduced hours, 365 assigned original days, 24 TSAM groups, 84 representatives
- representative structure: each representative has exactly 24 hourly rows
- assignment integrity: every original day is assigned exactly once
- weight consistency: representative weights match the number of assigned original days

In [298]:
expected_reduced_hours: Final[int] = 84 * 24
expected_original_days: Final[int] = 365
expected_group_count: Final[int] = 12 * 2
expected_representative_days: Final[int] = 12 * (
    WORKING_CLUSTERS_PER_MONTH + NON_WORKING_CLUSTERS_PER_MONTH
)

# Expected object sizes.
assert len(reduced_hourly_df) == expected_reduced_hours
assert len(day_assignments_df) == expected_original_days
assert len(tsam_results_by_group) == expected_group_count
assert len(representative_days) == expected_representative_days

# Representative structure.
assert reduced_hourly_df["representative_id"].nunique() == expected_representative_days
assert (reduced_hourly_df["representative_id"].value_counts() == 24).all()

representatives_per_month = (
    representative_days
    .groupby(["month", "day_type"])
    .size()
    .unstack(fill_value=0)
)
assert (representatives_per_month["working"] == WORKING_CLUSTERS_PER_MONTH).all()
assert (representatives_per_month["non-working"] == NON_WORKING_CLUSTERS_PER_MONTH).all()

# Assignment integrity.
assert day_assignments_df.index.is_unique
assert day_assignments_df["cluster_id"].notna().all()
assert set(day_assignments_df["representative_id"]) == set(
    reduced_hourly_df["representative_id"]
)

# Weight consistency.
assert representative_days["cluster_weight"].sum() == expected_original_days

assigned_day_counts = day_assignments_df["representative_id"].value_counts()
representative_weights = representative_days.set_index("representative_id")[
    "cluster_weight"
]
assert assigned_day_counts.sort_index().equals(representative_weights.sort_index())

print("The validation has been passed!")

The validation has been passed!


The reduced chronology now has the expected shape and every original day maps to exactly one representative day.

## Looking At The Results (TO ADD in subsections of this section what each (the first 3 subsections) dataframe is needed for later after we use it in plotting)
[API reference](https://tsam.readthedocs.io/en/latest/api/tsam/result/#tsam.result.AggregationResult)

Let's inspect four tables we got from the previous step:


### Representative Days

In [299]:
representative_days.head()


,selected_medoid_date,representative_id,group_id,month,day_type,cluster_id,cluster_weight
0,2025-01-02,2025_1_working_c4,2025_1_working,1,working,4,2
1,2025-01-04,2025_1_non-working_c0,2025_1_non-working,1,non-working,0,6
2,2025-01-06,2025_1_working_c2,2025_1_working,1,working,2,3
3,2025-01-08,2025_1_working_c3,2025_1_working,1,working,3,7
4,2025-01-16,2025_1_working_c1,2025_1_working,1,working,1,5


In [300]:
representative_days.shape

(84, 7)

This confirms:
- 84 selected medoid days
- one row per representative_id
- cluster_weight attached
- selected_medoid date visible

### Day Assignments


In [301]:
day_assignments_df.head()

,cluster_id,cluster_weight,group_id,representative_id
date,,,,
2025-01-01,2,3,2025_1_working,2025_1_working_c2
2025-01-02,4,2,2025_1_working,2025_1_working_c4
2025-01-03,4,2,2025_1_working,2025_1_working_c4
2025-01-04,0,6,2025_1_non-working,2025_1_non-working_c0
2025-01-05,0,6,2025_1_non-working,2025_1_non-working_c0


This confirms:
- 365 original days
- each original day has cluster_id, cluster_weight
- each original day has representative_id

### Reduced Hourly Data


In [302]:
reduced_hourly_df.head()

,demand_DE,DE_solar_2025,DE_onwind_2025,DE_ror_2025,DE_hydro_2025,date,month,day_of_month,weekday,day_type,group_id,group_day_number,cluster_id,cluster_weight,representative_id
snapshot,,,,,,,,,,,,,,,
2025-01-02 00:00:00,29735,0.0,0.883868,0.577233,67,2025-01-02,1,2,Thursday,working,2025_1_working,2,4,2,2025_1_working_c4
2025-01-02 01:00:00,29543,0.0,0.856581,0.576092,67,2025-01-02,1,2,Thursday,working,2025_1_working,2,4,2,2025_1_working_c4
2025-01-02 02:00:00,29188,0.0,0.812696,0.577028,67,2025-01-02,1,2,Thursday,working,2025_1_working,2,4,2,2025_1_working_c4
2025-01-02 03:00:00,30366,0.0,0.771761,0.569752,66,2025-01-02,1,2,Thursday,working,2025_1_working,2,4,2,2025_1_working_c4
2025-01-02 04:00:00,32484,0.0,0.729472,0.571890,66,2025-01-02,1,2,Thursday,working,2025_1_working,2,4,2,2025_1_working_c4


In [303]:
reduced_hourly_df.shape

(2016, 15)

### This confirms:
- 2016 hourly rows
- 24 rows per representative day
- weights attached to every hour

### One Raw TSAM Result
We have 24 groups, and looking at each will make this notebook unnecessarily long, so let's take a look at just one:

In [304]:
example_group_id = group_ids[0]
example_result = tsam_results_by_group[example_group_id]
print(f"The results of the group we are inspecting here is – '{example_group_id}'")

The results of the group we are inspecting here is – '2025_10_non-working'


#### Cluster representatives output table

In [305]:
example_result.cluster_representatives.head()

DE_hydro_2025  DE_onwind_2025  DE_ror_2025  DE_solar_2025  \
  timestep                                                              
0 0                 114.0        0.108620     0.541777            0.0   
  1                 114.0        0.113846     0.533667            0.0   
  2                 113.0        0.118070     0.536874            0.0   
  3                 113.0        0.126059     0.538439            0.0   
  4                 112.0        0.133861     0.548685            0.0   

            demand_DE  
  timestep             
0 0           29671.0  
  1           29398.0  
  2           29497.0  
  3           30315.0  
  4           31283.0

The index has the shape `(cluster number, hour of day)`. With `representation="medoid"`, each representative profile is based on a real day selected from its cluster.


#### Cluster Assignments


In [306]:
example_result.cluster_assignments

array([1, 1, 0, 0, 0, 0, 1, 1])

In [307]:
example_result.cluster_assignments.shape

(8,)

This 8 element-long array tells us which cluster each original day belongs to.

**Example:** the first and second days belong to cluster `1`; the third, fourth, fifth, sixth days belong to cluster `0`.


#### Assignments


In [308]:
example_result.assignments.shape

(192, 3)

In [309]:
example_result.assignments.head()

,period_idx,timestep_idx,cluster_idx
snapshot,,,
2025-10-04 00:00:00,0,0,1
2025-10-04 01:00:00,0,1,1
2025-10-04 02:00:00,0,2,1
2025-10-04 03:00:00,0,3,1
2025-10-04 04:00:00,0,4,1


This is the same assignment data with extra indexing columns.

Columns:
- `period_idx`: chronological period number. With `period_duration=24`, one period equals one day.
- `timestep_idx`: hour inside the period. With 24-hour periods, it ranges from `0` to `23`.
- `cluster_idx`: cluster assigned to that period.


#### Cluster weights

In [310]:
example_result.cluster_weights

{np.int64(0): np.int64(4), np.int64(1): np.int64(4)}

Cluster weights show how many original days each representative day stands for.

**Example:** if cluster `0` has weight `4`, its representative day replaces 4 original days.


#### Reconstructed data

In [311]:
example_result.reconstructed.shape

(192, 5)

In [312]:
example_result.reconstructed.head()

,DE_hydro_2025,DE_onwind_2025,DE_ror_2025,DE_solar_2025,demand_DE
snapshot,,,,,
2025-10-04 00:00:00,88.0,0.576077,0.555585,0.0,26067.0
2025-10-04 01:00:00,88.0,0.607647,0.555628,0.0,26359.0
2025-10-04 02:00:00,88.0,0.636138,0.551815,0.0,27340.0
2025-10-04 03:00:00,88.0,0.679335,0.547300,0.0,29397.0
2025-10-04 04:00:00,88.0,0.731166,0.545402,0.0,33870.0


The reconstructed dataset has the original group hourly data. In this case our group is '2025_10_non-working', which in this case means that there is 192 hours of non-working days in october 2025.  Each original day is replaced by the representative day of its assigned cluster.


#### Residuals

In [313]:
example_result.residuals.head()

,DE_hydro_2025,DE_onwind_2025,DE_ror_2025,DE_solar_2025,demand_DE
snapshot,,,,,
2025-10-04 00:00:00,0.0,0.0,0.0,0.0,0.000000e+00
2025-10-04 01:00:00,0.0,0.0,0.0,0.0,0.000000e+00
2025-10-04 02:00:00,0.0,0.0,0.0,0.0,-3.637979e-12
2025-10-04 03:00:00,0.0,0.0,0.0,0.0,0.000000e+00
2025-10-04 04:00:00,0.0,0.0,0.0,0.0,0.000000e+00


Residuals are pointwise errors between the original full-resolution data and the reconstructed data.


#### Metrics
[API reference](https://tsam.readthedocs.io/en/latest/api/tsam/result/#tsam.result.AccuracyMetrics)

In [314]:
example_result.accuracy

AccuracyMetrics(
  rmse=0.2122 (weighted),
  mae=0.1306 (weighted),
  rmse_duration=0.1375 (weighted)
)

Accuracy metrics comparing aggregated/reconstructed data against the original data:
- `RMSE`: penalizes large errors more heavily.
- `MAE`: treats errors more evenly.
- `RMSE_duration`: measures error in the duration curve, where value distribution matters more than timing.


Inspect the errors per column.


In [315]:
example_result.accuracy.rmse

DE_hydro_2025     0.345127
DE_onwind_2025    0.122207
DE_ror_2025       0.179367
DE_solar_2025     0.134533
demand_DE         0.202217
Name: RMSE, dtype: float64

In [316]:
example_result.accuracy.mae

DE_hydro_2025     0.240625
DE_onwind_2025    0.078945
DE_ror_2025       0.136518
DE_solar_2025     0.056239
demand_DE         0.140878
Name: MAE, dtype: float64

In [317]:
example_result.accuracy.rmse_duration

DE_hydro_2025     0.210324
DE_onwind_2025    0.054296
DE_ror_2025       0.142187
DE_solar_2025     0.113232
demand_DE         0.119623
Name: RMSE_duration, dtype: float64

#### Extremes Comparison
Compare annual **min/max** values to see how well the reconstructed data preserves extreme values.


In [318]:
original: pd.DataFrame = data_DE
reconstructed: pd.DataFrame = example_result.reconstructed

Calculate min/max errors.


In [319]:
def build_extreme_value_comparison(
    original: pd.DataFrame,
    reconstructed: pd.DataFrame,
) -> pd.DataFrame:
    """Compare annual min/max values between original and reconstructed series."""
    comparison = pd.DataFrame(
        {
            "original_min": original.min(axis=0),
            "reconstructed_min": reconstructed.min(axis=0),
            "original_max": original.max(axis=0),
            "reconstructed_max": reconstructed.max(axis=0),
        }
    )

    comparison["min_error"] = (
        comparison["reconstructed_min"] - comparison["original_min"]
    )
    comparison["max_error"] = (
        comparison["reconstructed_max"] - comparison["original_max"]
    )
    comparison["max_error_%"] = (
        comparison["max_error"].div(comparison["original_max"]) * 100
    )

    return comparison


extreme_value_comparison = build_extreme_value_comparison(
    original, reconstructed
)
extreme_value_comparison


,original_min,reconstructed_min,original_max,reconstructed_max,min_error,max_error,max_error_%
DE_hydro_2025,50.000000,88.000000,463.000000,114.000000,38.000000,-349.000000,-75.377970
DE_onwind_2025,0.000000,0.036670,0.914319,0.876295,0.036670,-0.038024,-4.158761
DE_ror_2025,0.344618,0.522609,1.000000,0.621188,0.177991,-0.378812,-37.881200
DE_solar_2025,0.000000,0.000000,0.766724,0.240596,0.000000,-0.526128,-68.620199
demand_DE,22625.000000,26067.000000,53574.000000,41296.000000,3442.000000,-12278.000000,-22.917833


# Plotting Results
[API reference](https://tsam.readthedocs.io/en/latest/api/tsam/plot/#tsamplot)

Tables are useful, but plots make the aggregation behavior easier to inspect. TSAM provides most of these plots out of the box.


In [ ]:
# Setting visual theme here for persistence
plotly_theme: Final[str] = "ggplot2"

The dataset contains features on very different scales: capacity factors are in the `0-1` range, while demand is much larger. Plotting them separately keeps the charts readable.


In [ ]:
# These columns share the same 0.0-1.0 capacity-factor scale.
CF_features: Final[list[str]] = [
    "DE_onwind_2025",
    "DE_ror_2025",
    "DE_solar_2025",
]

## Cluster Representatives
These line charts show the **representative daily profiles** for each cluster.


Capacity-factor features first.


In [ ]:
fig = result.plot.cluster_representatives(
    columns=CF_features, title="Cluster representative profiles (CFs)"
)
fig.update_layout(template=plotly_theme)

Hydro inflow.


In [ ]:
fig = result.plot.cluster_representatives(
    columns=["DE_hydro_2025"],
    title="Cluster representative profiles (DE_hydro_2025)",
)
fig.update_layout(template=plotly_theme)

Demand.


In [ ]:
fig = result.plot.cluster_representatives(
    columns=["demand_DE"], title="Cluster representative profiles (demand_DE)"
)
fig.update_layout(template=plotly_theme)

## Cluster Members
This plot groups original periods by cluster and highlights the representative period.


In [ ]:
fig = result.plot.cluster_members()
fig.update_layout(template=plotly_theme)

Use the slider to inspect each cluster.


## Cluster Weights
This bar chart shows each cluster weight: **how many original days each representative day replaces**.


In [ ]:
fig = result.plot.cluster_weights()
fig.update_layout(template=plotly_theme)

## Cluster Accuracy
This bar chart shows accuracy metrics for the reduced/reconstructed data against the original full-resolution data.


In [ ]:
fig = result.plot.accuracy()
fig.update_layout(template=plotly_theme)

## Full-Resolution Comparison
Compare the original and reconstructed time series.


Capacity-factor features.


In [ ]:
fig = result.plot.compare(
    columns=CF_features, title="Original vs Reconstructed (CFs)"
)
fig.update_layout(template=plotly_theme)

Hydro inflow.


In [ ]:
fig = result.plot.compare(
    columns=["DE_hydro_2025"], title="Original vs Reconstructed (DE_hydro_2025)"
)
fig.update_layout(template=plotly_theme)

Demand.


In [ ]:
fig = result.plot.compare(columns=["demand_DE"])
fig.update_layout(template=plotly_theme)

## Residuals
These line charts show the error between original and reconstructed values over time.


Capacity-factor features.


In [ ]:
fig = result.plot.residuals(columns=CF_features)
fig.update_layout(template="ggplot2")

Hydro inflow.


In [ ]:
fig = result.plot.residuals(columns=["DE_hydro_2025"])
fig.update_layout(template="ggplot2")

Demand.


In [ ]:
fig = result.plot.residuals(columns=["demand_DE"])
fig.update_layout(template="ggplot2")

It is also useful to inspect which original days were assigned to each cluster. TSAM does not provide this calendar view directly, so we build it manually.


## Calendar Heatmap Of Original Days By Assigned Cluster
Each day is colored by its assigned cluster.


First, prepare the data and chart configuration.

**Note:** These preparation cells are code-heavy and are designed to work with different clustering configurations. You can skip them and jump directly to the chart output.


### Data Preparation Code (Feel Free To Skip)


In [ ]:
# One row per original day, with the assigned TSAM cluster for that day.
day_to_cluster: pd.DataFrame = pd.DataFrame(
    data=result.cluster_assignments,
    index=pd.date_range(start="2025-01-01", end="2025-12-31", freq="D"),
    columns=["cluster"],
)

# Split into 12 month-level DataFrames for calendar plotting.
monthly_day_to_cluster: list[pd.DataFrame] = [
    day_to_cluster[day_to_cluster.index.month == month]
    for month in range(1, 13)
]


def prep_month_for_plotting(
    month: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Add calendar coordinates and return a week-row x weekday cluster grid."""
    month = month.copy()
    month["day"] = month.index.day
    month["weekday"] = month.index.day_name()
    month["weekday_num"] = month.index.weekday

    first_weekday = month.index.min().weekday()
    month["week_row"] = (month["day"] + first_weekday - 1) // 7

    calendar_grid = month.pivot(
        index="week_row",
        columns="weekday_num",
        values="cluster",
    )
    return month, calendar_grid


processed_mo_to_clusters: list[tuple[pd.DataFrame, pd.DataFrame]] = [
    prep_month_for_plotting(month) for month in monthly_day_to_cluster
]

### Plotting Setup Code (Feel Free To Skip)
Single-month calendar plotting code.


In [ ]:
def plot_calendar_month(
    month_data: pd.DataFrame, month_grid: pd.DataFrame
) -> go.Heatmap:
    """Build one month-calendar heatmap trace colored by assigned cluster."""
    weekday_labels = ["Mo", "Tu", "We", "Th", "Fr", "Sa", "Su"]
    calendar_grid = month_grid.reindex(columns=range(7))

    day_number_grid = month_data.pivot(
        index="week_row",
        columns="weekday_num",
        values="day",
    ).reindex(columns=range(7))
    day_text_grid = day_number_grid.map(
        lambda value: "" if pd.isna(value) else str(int(value))
    )

    cluster_ids: list[int] = sorted(day_to_cluster["cluster"].unique())
    zmin = min(cluster_ids) - 0.5
    zmax = max(cluster_ids) + 0.5
    palette = (
        px.colors.qualitative.Plotly
        + px.colors.qualitative.Set3
        + px.colors.qualitative.Dark24
    )
    cluster_colors = {
        cluster_id: palette[i % len(palette)]
        for i, cluster_id in enumerate(cluster_ids)
    }

    # Duplicate each color stop so integer cluster IDs render as discrete categories.
    colorscale: list[tuple[float, str]] = []
    for cluster_id in cluster_ids:
        color = cluster_colors[cluster_id]
        colorscale.append(((cluster_id - 0.5 - zmin) / (zmax - zmin), color))
        colorscale.append(((cluster_id + 0.5 - zmin) / (zmax - zmin), color))

    weekday_hover_labels = pd.DataFrame(
        [weekday_labels] * len(calendar_grid),
        index=calendar_grid.index,
        columns=range(7),
    )

    return go.Heatmap(
        z=calendar_grid.to_numpy(),
        x=list(range(7)),
        y=calendar_grid.index,
        text=day_text_grid.to_numpy(),
        customdata=weekday_hover_labels.to_numpy(),
        texttemplate="%{text}",
        textfont={"size": 13, "color": "#333"},
        colorscale=colorscale,
        zmin=zmin,
        zmax=zmax,
        hovertemplate="Day %{text}<br>%{customdata}<br>Cluster %{z}<extra></extra>",
        xgap=2,
        ygap=2,
        colorbar={
            "title": "Cluster",
            "tickmode": "array",
            "tickvals": cluster_ids,
            "ticks": "",
        },
        showscale=False,
    )


figs: list[go.Heatmap] = [
    plot_calendar_month(month_data, month_grid)
    for month_data, month_grid in processed_mo_to_clusters
]

Full-year calendar plotting code.


In [ ]:
ROWS: Final[int] = 4
COLS: Final[int] = 3
WEEKDAY_LABELS: Final[list[str]] = ["Mo", "Tu", "We", "Th", "Fr", "Sa", "Su"]
MONTH_TITLES: Final[list[str]] = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December",
]

fig = make_subplots(rows=ROWS, cols=COLS, subplot_titles=MONTH_TITLES)
subplot_positions: list[tuple[int, int]] = [
    (row, col) for row in range(1, ROWS + 1) for col in range(1, COLS + 1)
]
fig.add_traces(
    data=figs,
    rows=[row for row, _ in subplot_positions],
    cols=[col for _, col in subplot_positions],
)

# Weekend cells keep their cluster fill color; the border is the weekend marker.
for (month_data, _), (row, col) in zip(
    processed_mo_to_clusters, subplot_positions
):
    weekend_days = month_data[month_data["weekday_num"].isin([5, 6])]
    for _, day in weekend_days.iterrows():
        fig.add_shape(
            type="rect",
            x0=day["weekday_num"] - 0.5,
            x1=day["weekday_num"] + 0.5,
            y0=day["week_row"] - 0.5,
            y1=day["week_row"] + 0.5,
            line=dict(color="rgba(35, 35, 35, 0.75)", width=2),
            fillcolor="rgba(0, 0, 0, 0)",
            layer="above",
            row=row,
            col=col,
        )

fig.update_layout(
    title={
        "text": "2025 clusters calendar",
        "x": 0.5,
        "font_size": 20,
        "y": 0.99,
    },
    template=plotly_theme,
    width=1000,
    height=900,
    plot_bgcolor="white",
    margin={"l": 20, "r": 20, "t": 120, "b": 20},
)

fig.update_annotations(yshift=25)
fig.update_xaxes(
    side="top",
    tickmode="array",
    tickvals=list(range(7)),
    ticktext=WEEKDAY_LABELS,
    showticklabels=True,
    ticks="",
    showline=False,
    showgrid=False,
    zeroline=False,
)
fig.update_yaxes(
    ticks="",
    showline=False,
    autorange="reversed",
    showgrid=False,
    zeroline=False,
    showticklabels=False,
)

fig.show()

## Month-By-Cluster Heatmap
This heatmap summarizes how many days from each month were assigned to each cluster.


### Data Prep Code (Feel Free To Skip)


In [ ]:
month_cluster_counts = (
    day_to_cluster.assign(month=day_to_cluster.index.month_name())
    .groupby("month")["cluster"]
    .value_counts()
)

Pivot the data into a month-by-cluster matrix.


In [ ]:
month_cluster_counts = month_cluster_counts.unstack(
    level="cluster", fill_value=0
).sort_index(axis=1)

Sort months in calendar order.


In [ ]:
month_order: Final[list[str]] = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December",
]

month_cluster_counts.index = pd.CategoricalIndex(
    month_cluster_counts.index,
    categories=month_order,
    ordered=True,
)
month_cluster_counts.sort_index(inplace=True)

month_cluster_pct = (
    month_cluster_counts.div(month_cluster_counts.sum(axis=1), axis=0)
    .mul(100)
    .round(1)
)

### Plotting Code (Feel Free To Skip)


In [ ]:
fig = px.imshow(
    month_cluster_pct,
    text_auto=True,
    labels={"x": "Cluster", "y": "Month", "color": "% of month"},
)

fig.update_traces(
    customdata=month_cluster_counts.to_numpy()[:, :, None],
    texttemplate="%{z:.1f}%",
    hovertemplate=(
        "Month: %{y}<br>"
        "Cluster: %{x}<br>"
        "Share of month: %{z:.1f}%<br>"
        "Days assigned: %{customdata[0]:.0f}"
        "<extra></extra>"
    ),
)
fig.update_layout(
    title={
        "text": "Share of each month assigned to each cluster",
        "x": 0.5,
        "font_size": 20,
    },
    coloraxis_showscale=False,
)
fig.show()

# Saving Results
TSAM outputs are pandas DataFrames, so they can be saved in any format pandas supports.

[pandas-supported file formats](https://pandas.pydata.org/docs/user_guide/io.html)

Here, we save the reconstructed 8760-hour dataset to a `.csv` file.


In [ ]:
import os

# Create a directory where the results are going to be stored
output_path: Path = cur_location / "outputs"
os.makedirs(output_path, exist_ok=True)

# Save to disk in a specified path under file_name
file_name = "results.csv"
result.reconstructed.to_csv(output_path / file_name)